### README

From rMATS alternative splicing analysis, do:
- EVENT CLASSIFICATION: stratify NAGNAG events into two groups: those with significant changes upon FAM32A knockdown (siFAM32A) and those showing no significant alteration;
- FEATURE CHARACTERISATION: extract and investigate the feature differences between the regulated and non-regulated groups;
- FEATURE SELECTION: select features derived from the primary sequence to train a downstream machine learning model.

NOTE: Replace /PATH/TO/... with the actual paths on your system.


### Requirements

In [ ]:
# Python libraries
"""
biopython==1.85
logomaker==0.8.6
matplotlib==3.10.0
numpy==2.0.2
pandas==2.2.3
scipy==1.15.2
seaborn==0.13.2
statsmodels==0.14.4
"""

# Outer softwares
"""
SVM-BPfinder-3M: Corvelo A, Hallegger M, Smith CW, Eyras E. Genome-wide association between branch point properties and alternative splicing. PLoS Comput Biol. 2010 Nov 24;6(11):e1001016. doi: 10.1371/journal.pcbi.1001016. PMID: 21124863; PMCID: PMC2991248.
maxentscan: Yeo G, Burge CB. Maximum entropy modeling of short sequence motifs with applications to RNA splicing signals. J Comput Biol. 2004;11(2-3):377-94. doi: 10.1089/1066527041410418. PMID: 15285897.
"""

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from Bio import SeqIO
from collections import defaultdict
from Bio.SeqUtils import gc_fraction

import gzip
import subprocess
import io
import tempfile
import os

from scipy import stats
from statsmodels.stats.multitest import multipletests
from itertools import combinations


import matplotlib.pyplot as plt
import seaborn as sns
import logomaker


#Outer softwares
bpfinder = '/PATH/TO/SVM-BPfinder-3M/svm_bpfinder.py'
maxentscan = '/PATH/TO/maxEntScan/fordownload'

### Constants

In [ ]:
# Input: rMATS results A3SS (siCTRL_VS_siFAM32A)
RMATS_PATH = '/PATH/TO/A3SS.MATS.JCEC.txt' 

# Input: expression Getmm file
GETMM_PATH = '/PATH/TO/GeTMM.txt'

# Input: genome fasta file
FASTA_PATH = '/PATH/TO/Homo_sapiens.GRCh38.dna.primary_assembly.fa.gz' 

# Output: set directory
OUTPUT_PATH = '/PATH/TO/OUTPUT'

# Sig events filters
READS_MIN = 10
FDR_MIN = 0.01
DPSI_REG = 0.15
DPSI_NONREG = 0.05

### Functions

#### Classify events

In [ ]:
# SUPPORTING FUNCTIONS TO DEFINE REGULATED VS NON REGULATED NAGNAG

# calculate mean from string cols with numbers separated by comma
def get_mean_str(series):

    return series.str.split(',') \
                 .apply(lambda x: np.nanmean(pd.to_numeric(x, errors='coerce')))

# calculate distance between two acceptor sites
def get_distance(row):
    if row['strand'] == '+':
        return row['shortES'] - row['longExonStart_0base']
    else:
        return row['longExonEnd'] - row['shortEE']
    
    
# calculate mean counts
def read_counts(df, sample_number='1'):

    total = pd.Series(0.0, index=df.index)

    for col_prefix in ['IJC_SAMPLE_', 'SJC_SAMPLE_']:
        col = df[f'{col_prefix}{sample_number}'].astype(str)

        total += get_mean_str(col)

    return total


# equivalence test tost
def tost_pvals(df, dpsi_nonreg=0.05, var_eps=1e-18, jitter=1e-18, random_state=42):

    # Create RNG (local, reproducible)
    rng = np.random.default_rng(random_state)

    pvals = []

    for idx, row in df.iterrows():
        try:
            # Convert strings to array of floats
            inc1 = np.array([float(i) for i in str(row["IncLevel1"]).split(",") if i.strip() != "NA"], dtype=float)
            inc2 = np.array([float(i) for i in str(row["IncLevel2"]).split(",") if i.strip() != "NA"], dtype=float)

            # Skip if not enough replicates
            if inc1.size <= 1 or inc2.size <= 1:
                pvals.append(np.nan)
                continue

            # Add small jitter to group(s) with near-zero variance
            if np.var(inc1) <= var_eps:
                inc1 = inc1 + rng.normal(0, jitter, size=len(inc1))

            if np.var(inc2) <= var_eps:
                inc2 = inc2 + rng.normal(0, jitter, size=len(inc2))

            # Unpaired two-sample t-tests
            _, p_greater = stats.ttest_ind(np.array(inc1) + dpsi_nonreg, inc2, alternative='greater')
            _, p_less = stats.ttest_ind(np.array(inc1) - dpsi_nonreg, inc2, alternative='less')

            # Maximum p-value
            pvals.append(max(p_less, p_greater))

        except Exception as e:
            print(idx, row["IncLevel1"], row["IncLevel2"], e)
            pvals.append(np.nan)

    return pd.Series(pvals, index=df.index)



# adjust pvalues for multiple comparisons
def get_fdr(pvals, method='fdr_bh'):

    # Ensure we only adjust non-NaN values
    pvals_nonan = pvals.dropna()
    
    # Apply FDR correction
    _, fdr_pvals, _, _ = multipletests(pvals_nonan, method=method)
    
    # Create a Series with the same index as the original
    fdr_series = pd.Series(index=pvals.index, dtype=float)
    fdr_series[pvals_nonan.index] = fdr_pvals
    
    return fdr_series



In [ ]:
# DEFINE REGULATED VS NON REGULATED NAGNAG

def classify_regulation(df, fdr_min=0.01, dpsi_reg=0.15, dpsi_nonreg=0.05):
    
    df = df.copy()

    classification = np.where(
        (df['FDR'] <= fdr_min) & (df['IncLevelDifference'].abs() >= dpsi_reg), 1,
        np.where((df['TOST_FDR'] <= fdr_min) & (df['IncLevelDifference'].abs() <= dpsi_nonreg), 0, 2)
    )

    return pd.Series(classification, index=df.index)

#### Gene expression

In [ ]:
# GET INFORMATION FROM GENE EXPRESSION

def get_getmm(getmm_path, input_df, gene_col="GeneID", condition="siCTRL"):
    
    # Read GeTMM file
    getmm_df = pd.read_table(getmm_path, index_col=0)

    # Cols from specified condition
    condition_cols = [col for col in getmm_df.columns if col.startswith(condition + "_")]

    if not condition_cols:
        raise ValueError(f"Condition '{condition}' not found in GeTMM data.")

    # Mean of samples from the specified condition
    mean_series = getmm_df[condition_cols].mean(axis=1)

    # Return expression of specified condition for each gene in the input df
    return input_df[gene_col].map(mean_series)



#### Sequence related features

In [ ]:
# Create dict with position lists per chromosome
def build_positions(df):
    positions = defaultdict(list)
    for row in df.itertuples():
        positions[row.chr].append((
            row.Index, row.strand, 
            row.flankingEE, row.shortES,
            row.shortEE, row.flankingES
        ))
    return positions

# Get GC content and sequence from A3SS events: flexon, intron, exon
def compute_gc_and_seq(records, positions, five_prime_offset=5, three_prime_offset=5, A3SS_length=3):
    
    seq_dict = {}
    gc_flexon_dict = {}
    gc_intron_dict = {}
    gc_exon_dict = {}

    for chromosome, entries in positions.items():
        long_seq = records[chromosome].seq
        for (Index, strand, flankingEE, shortES, shortEE, flankingES) in entries:
            if strand == "+":
                short_seq = long_seq[flankingEE-five_prime_offset : shortES+three_prime_offset]
                flexon_gc = gc_fraction(long_seq[flankingES:flankingEE])
                intron_gc  = gc_fraction(long_seq[flankingEE : shortES-A3SS_length])
                exon_gc    = gc_fraction(long_seq[shortES-A3SS_length : shortEE])
            else:
                short_seq = long_seq[shortEE-three_prime_offset : flankingES+five_prime_offset].reverse_complement()
                flexon_gc = gc_fraction(long_seq[flankingES:flankingEE])
                intron_gc = gc_fraction(long_seq[shortEE+A3SS_length : flankingES])
                exon_gc   = gc_fraction(long_seq[shortES : shortEE+A3SS_length])
            
            seq_dict[Index] = str(short_seq)
            gc_flexon_dict[Index] = float(flexon_gc)
            gc_intron_dict[Index] = float(intron_gc)
            gc_exon_dict[Index] = float(exon_gc)
    
    return seq_dict, gc_flexon_dict, gc_intron_dict, gc_exon_dict



# Calculates flexon, intron and exon lenghts considering the use of the proximal site
def get_lengths(df):

    # Flanking exon
    Len_flexon = df['flankingEE'] - df['flankingES']
    
    # Intron
    Len_intron = np.where(
        df['strand'] == '+',
        df['shortES'] - df['flankingEE'],  # '+' strand
        df['flankingES'] - df['shortEE']   # '-' strand
    )
    Len_intron = pd.Series(Len_intron, index=df.index)
    
    # Exon
    Len_exon = df['longExonEnd'] - df['longExonStart_0base']
    
    return Len_flexon, Len_intron, Len_exon



# get splice site scores
def maxentscan_score_3ss(sequences):
    
    # Create a temp file for input sequences
    with tempfile.NamedTemporaryFile(mode='w+', delete=False) as temp_input:
        for seq in sequences:
            temp_input.write(seq + '\n')
        temp_input_name = temp_input.name

    try:
        script_path = os.path.join(maxentscan, 'score3.pl')

        # Run score3.pl from maxentscan_dir to ensure matrix files found
        result = subprocess.run(
            ['perl', script_path, temp_input_name],
            cwd=maxentscan,
            capture_output=True,
            text=True,
            check=True
        )

        # Parse scores: each line is "<sequence>\t<score>"
        scores = [float(line.strip().split('\t')[1]) for line in result.stdout.strip().split('\n')]

    finally:
        # Clean up temp file
        os.remove(temp_input_name)

    return scores

# writes a fasta file with intron sequece, each intron is an identifier
def write_fasta(seq_dict, fasta_filename, trim=(5, 8)):
    # trim: number of bases do remove from 5' and 3' end
    start_trim, end_trim = trim

    with open(fasta_filename, "w") as fasta_file:
        for intron_id, seq in seq_dict.items():
            seq = str(seq)
            
            # ignore seq shorter that trim
            if len(seq) <= start_trim + end_trim:
                continue
            
            trimmed_seq = seq[start_trim:-end_trim] if end_trim > 0 else seq[start_trim:]
            fasta_file.write(f">{intron_id}\n{trimmed_seq}\n")

# get BPfinder output from fasta file
# output is a dataframe showing only the best prediction of BP per intron
def find_bp(fasta_filename):
    
    # Run SVM-BPfinder
    cmd = f"{bpfinder} -i {fasta_filename} -s Hsap -l 100 -d 10"
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    
    # Check for errors
    if result.returncode != 0:
        print("Error running SVM-BPfinder:", result.stderr)
        return None

    # Convert output into a DataFrame
    svm_output = pd.read_csv(io.StringIO(result.stdout), sep="\t")
    
    # Ensure `svm_scr` is numeric
    svm_output["svm_scr"] = pd.to_numeric(svm_output["svm_scr"], errors="coerce")

    # Select the row with the highest `svm_scr` for each intron
    svm_output_best = svm_output.loc[svm_output.groupby("seq_id")["svm_scr"].idxmax()]
    
    return svm_output_best



#### Plots

In [ ]:
# stacked bar plot with percentages
def plot_stacked_bar(
    data_df,
    column,
    hue=None,
    color_dict=None,
    title=None,
    ylabel='Count',
    figsize=(3,5),
    filename=None,
    sort_stack=False 
):
   
    df = data_df.copy() 
    
    # check if hue is present
    if hue is None:
        df['_'] = ''
        hue = '_'
    
    groups = df.groupby(hue)
    x_positions = range(len(groups))

    fig, ax = plt.subplots(figsize=figsize)

    # create each bar
    for x_pos, (hue_val, group) in zip(x_positions, groups):
        counts = group[column].value_counts()
        if sort_stack:
            counts = counts.sort_values(ascending=False)
        total = counts.sum()
        proportions = counts / total * 100

        bottom = 0
        for val, count in counts.items():
            pct = proportions[val]
            color = color_dict[val] if color_dict and val in color_dict else 'grey'

            ax.bar(x_pos, count, bottom=bottom, color=color)

            # add text with percentage and counts
            ax.text(
                x=x_pos,
                y=bottom + count/2,
                s=f"{pct:.1f}% ({count})",
                ha='center',
                va='center',
                color='black',
                fontsize=6,
                fontweight='bold',
                rotation=0
            )

            bottom += count

    
    # Add legend
    all_vals = pd.unique(df[column])
    handles = [plt.Rectangle((0,0),1,1,color=color_dict[val] if color_dict and val in color_dict else 'grey')
               for val in all_vals]
    labels = [str(val) for val in all_vals]
    ax.legend(handles, labels, loc='upper left', bbox_to_anchor=(1,1), fontsize=10)

    # Change layout
    # axis
    ax.set_xticks(list(x_positions))
    ax.set_xticklabels([str(hue_val) for hue_val, _ in groups])
    ax.set_ylabel(ylabel)
    ax.set_xlabel(hue)
    # title
    ax.set_title(title or f'Stacked bar plot of {column}')
    # spines
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.tight_layout()

    if filename:
        plt.savefig(filename, transparent=True)

    plt.show()


In [ ]:
# plot sequence related features logo (z-score)
def plot_logo_zscore(
    data_df,
    col_prefix,
    hue=None,
    p_bg=0.25,
    filename=None,
    ylim=None
):

    df = data_df.copy()

    # Select base columns
    base_cols = [c for c in df.columns if col_prefix in c and 'base' in c]
    df = df[base_cols + ([hue] if hue is not None else [])]

    # Build sequence string
    df[col_prefix] = df[base_cols].astype(str).agg("".join, axis=1)

    # Define groups
    if hue is None:
        groups = [(None, df)]
    else:
        groups = [(val, sub_df) for val, sub_df in df.groupby(hue)]

    for hue_value, sub_df in groups:
        seq_list = sub_df[col_prefix].tolist()

        if len(seq_list) == 0:
            continue

        # Count matrix
        counts_df = logomaker.alignment_to_matrix(seq_list, to_type="counts")

        # Frequency matrix
        freq_df = counts_df.div(counts_df.sum(axis=1), axis=0)

        # Z-score
        N = len(seq_list)
        std_error = np.sqrt(p_bg * (1 - p_bg) / N)
        zscore_df = (freq_df - p_bg) / std_error

        # Plot
        fig, ax = plt.subplots(figsize=(len(seq_list[0]) / 2, 5))
        logo = logomaker.Logo(
            zscore_df,
            ax=ax,
            shade_below=0.5,
            fade_below=0.5
        )

        # Style
        if ylim:
            ax.set_ylim(ylim)

        logo.style_spines(visible=False)
        logo.style_spines(spines=['left'], visible=True)
        logo.style_xticks(rotation=90, fmt='%d', anchor=0)

        ax.set_ylabel("Z-Score", labelpad=-1)
        ax.xaxis.set_ticks_position('none')
        ax.xaxis.set_tick_params(pad=-1)
        ax.set_xticks([])

        title = col_prefix if hue is None else f'{col_prefix} {hue}={hue_value}'
        ax.set_title(title)

        if filename:
            if hue is not None:
                # adiciona o valor do grupo ao nome do arquivo
                name, ext = filename.rsplit(".", 1)
                file_to_save = f"{name}_{hue_value}.{ext}"
            else:
                file_to_save = filename
            plt.savefig(file_to_save, bbox_inches='tight', transparent=True)
            
        plt.show()


In [ ]:
def plot_violin(
    df,
    column,
    hue,
    palette=None,
    figsize=(4, 3),
    filename=None
):
    

    if palette is None:
        palette = {1: 'firebrick', 0: 'royalblue', 2: 'grey'}

    # Identify unique groups
    hue_values = sorted(df[hue].dropna().unique())

    for g1, g2 in combinations(hue_values, 2):

        group1 = df.loc[df[hue] == g1, column]
        group2 = df.loc[df[hue] == g2, column]

        # Mann-Whitney U test
        if len(group1) > 1 and len(group2) > 1:
            _, p_value = stats.mannwhitneyu(group1, group2)
        else:
            p_value = np.nan

        # Plot
        plt.figure(figsize=figsize)

        sns.violinplot(
            data=df,
            y=column,
            hue=hue,
            split=True,
            palette=palette,
            hue_order=[g1, g2]
        )

        plt.title(f"{column}\np = {p_value:.3e}")
        plt.xlabel(hue)
        plt.ylabel(column)

        plt.tight_layout()

        if filename:
            plt.savefig(filename, transparent=True)

        plt.show()
        plt.close()


### Analysis

#### 1. Read rmats file

In [ ]:
# read rmats file
rmats_df = pd.read_table(RMATS_PATH, index_col=0)

#### 2. Classify events

In [ ]:
# Select only NAGNAG events
rmats_df['Distance'] = rmats_df.apply(get_distance, axis=1)
rmats_df = rmats_df[rmats_df['Distance']==3].copy()

# Calculate read counts for samples 1 and 2
rmats_df['MeanCounts_SAMPLE_1'] = read_counts(rmats_df, sample_number='1')
rmats_df['MeanCounts_SAMPLE_2'] = read_counts(rmats_df, sample_number='2')

# Filter events with enough read counts
rmats_df = rmats_df[(rmats_df['MeanCounts_SAMPLE_1'] >= READS_MIN) & (rmats_df['MeanCounts_SAMPLE_2'] >= READS_MIN)].copy()

# Calculate TOST p-values
rmats_df['TOST_pval'] = tost_pvals(rmats_df, dpsi_nonreg=DPSI_NONREG, var_eps=1e-18, jitter=1e-10, random_state=42)

# Adjust p-values for FDR
rmats_df['TOST_FDR'] = get_fdr(rmats_df['TOST_pval'], method='fdr_bh')

In [ ]:
# Classify events on nonregulated 0, regulated 1 or inconclusive 2
rmats_df['Regulation'] = classify_regulation(
    rmats_df,
    fdr_min=FDR_MIN,
    dpsi_reg=DPSI_REG,
    dpsi_nonreg=DPSI_NONREG,
)

In [ ]:
# Selects only regulated and non regulated events
rmats_df = rmats_df[(rmats_df['Regulation']==0) | (rmats_df['Regulation']==1)].copy()

#### 3. Add mean PSI and gene expression

In [ ]:
# Info from gene expression in cltr samples
rmats_df["siCTRL_expression"] = get_getmm(
    GETMM_PATH,
    rmats_df,
    gene_col="GeneID",
    condition="siCTRL"
)

# Info from gene expression in siFam32a samples
rmats_df["siFam32a_expression"] = get_getmm(
    GETMM_PATH,
    rmats_df,
    gene_col="GeneID",
    condition="siFam32a"
)

In [ ]:
# Mean inclusion level in ctrl samples
rmats_df['Mean_IncLevel_siCTRL'] = get_mean_str(rmats_df['IncLevel1'])

# Mean inclusion level in siFam32a samples
rmats_df['Mean_IncLevel_siFam32a'] = get_mean_str(rmats_df['IncLevel2'])

#### 4. Add sequence related features

In [ ]:
# ----------
# ADD FEATURES
# seq, GC and len
# ----------

# remove chr from chromosome name
rmats_df['chr'] = rmats_df['chr'].apply(lambda x: x[3:] if x.startswith('chr') else x) 

# load fasta
with gzip.open(FASTA_PATH, "rt") as handle:
    records = SeqIO.to_dict(SeqIO.parse(handle, 'fasta'))

# build positions
positions = build_positions(rmats_df)

# compute seqs and GC
seq_dict, gc_flexon_dict, gc_intron_dict, gc_exon_dict = compute_gc_and_seq(
    records, positions, five_prime_offset=5, three_prime_offset=5, A3SS_length=3
)


# add seq information to the dataframe
for i in range(5):  # exonic part of the 5' end
    col_name = f"5end_{i-5}_base"  
    rmats_df[col_name] = rmats_df.index.map(
        lambda idx: str(seq_dict[idx])[i] if idx in seq_dict and len(str(seq_dict[idx])) > i else None
    )

for i in range(5,12):  # intronic part of the 5' end
    col_name = f"5end_{i-4}_base"  
    rmats_df[col_name] = rmats_df.index.map(
        lambda idx: str(seq_dict[idx])[i] if idx in seq_dict and len(str(seq_dict[idx])) > i else None
    )

for i in range(-15,-8):  # intronic part of the 3' end
    col_name = f"3end_{i+8}_base"  
    rmats_df[col_name] = rmats_df.index.map(
        lambda idx: str(seq_dict[idx])[i] if idx in seq_dict and len(str(seq_dict[idx])) > i else None
    )

for i in range(-8,0):  # exonic part of the 3' end
    col_name = f"3end_{i+9}_base"  
    rmats_df[col_name] = rmats_df.index.map(
        lambda idx: str(seq_dict[idx])[i] if idx in seq_dict and len(str(seq_dict[idx])) > i else None
    )


# add GC content information to the dataframe
rmats_df['GC(flexon)'] = rmats_df.index.map(gc_flexon_dict)
rmats_df['GC(intron)'] = rmats_df.index.map(gc_intron_dict)
rmats_df['GC(exon)']   = rmats_df.index.map(gc_exon_dict)


# add length information to the dataframe
rmats_df['Len(flexon)'], rmats_df['Len(intron)'], rmats_df['Len(exon)'] = get_lengths(rmats_df)

In [ ]:
# ----------
# ADD FEATURES
# splice site scores
# ----------

# get list of proximal and distal 3'ss for stregth analysis
proximal_3ss_list = rmats_df.index.map(lambda idx: (seq_dict[idx][-28:-5]) if idx in seq_dict else None)
distal_3ss_list = rmats_df.index.map(lambda idx: (seq_dict[idx][-25:-2]) if idx in seq_dict else None)

#calculate maxentscan scores
rmats_df['proximal_3ss_score'] = maxentscan_score_3ss(proximal_3ss_list)
rmats_df['distal_3ss_score'] = maxentscan_score_3ss(distal_3ss_list)
rmats_df['delta_3ss_score'] = rmats_df['distal_3ss_score'] - rmats_df['proximal_3ss_score']


In [ ]:
# ----------
# ADD FEATURES
# bpfinder variables
# ----------


# Write separate FASTA files for short and long introns
write_fasta(seq_dict, "short_introns.fa", trim=(5, 8))
write_fasta(seq_dict, "long_introns.fa", trim=(5, 5))

# Run BPfinder
shortbp_df = find_bp('short_introns.fa')
longbp_df = find_bp('long_introns.fa')

# Merge information from BP if using proximal or distal sites
longbp_df['bp_same'] = 1
longbp_df['ss_dist'] = longbp_df['ss_dist']-3
longbp_df = longbp_df[['seq_id', 'ss_dist', 'bp_seq', 'bp_same']]
mergebp_df = shortbp_df.merge(longbp_df, on=['seq_id', 'ss_dist', 'bp_seq'], how='left')
mergebp_df['bp_same'] = mergebp_df['bp_same'].fillna(0)

# Split the 'bp_seq' column into 9 separate columns
mergebp_df['bp_seq'] = mergebp_df['bp_seq'].str.upper()
bpsplit_df = mergebp_df["bp_seq"].apply(lambda x: pd.Series(list(x)))
# Rename the columns for clarity
bpsplit_df.columns = [f"bp_{i+1}_base" for i in range(9)]
# Merge back with the bp DataFrame
mergebp_df = pd.concat([mergebp_df, bpsplit_df], axis=1)

# Merge BP information to the rmats dataframe
rmats_df['seq_id'] = rmats_df.index.tolist()
rmats_df = rmats_df.merge(mergebp_df, how="left", on="seq_id")
rmats_df = rmats_df.drop(columns=['bp_seq', 'seq_id'])

In [ ]:
# ----------
# ADJUST FEATURES
# change T for U
# ----------
base_cols = [col for col in rmats_df.columns if col.endswith('_base')]
rmats_df[base_cols] = rmats_df[base_cols].replace('T', 'U')

In [ ]:
# ----------
# FILTER DATAFRAME
# specific cols
# no nan info
# ----------

# Remove original cols from rmats software
rmats_df = rmats_df[rmats_df.columns[27:89]].copy()

# Remove events with nan
rmats_df = rmats_df.dropna()

# Reset index
rmats_df.reset_index(drop=True, inplace=True)

#### 5. Data exploration

In [ ]:
# check proportion of groups
plot_stacked_bar(
    data_df=rmats_df,
    column='Regulation',
    color_dict={1: 'firebrick', 0: 'royalblue'},
    title='Regulated vs Non-Regulated Introns',
    filename=f'{OUTPUT_PATH}/proportion_regulated_nonregulated.svg',
)

In [ ]:
# check proportion of categorical non-sequence feature
plot_stacked_bar(
    data_df=rmats_df,
    column='bp_same',
    hue='Regulation',
    color_dict={1: 'firebrick', 0: 'royalblue'},
    title="Same BP if using prox/distal 3'ss",
    filename=f'{OUTPUT_PATH}/proportion_bp_same.svg',
)

In [ ]:
# check proportion of sequence related features with sequence logo (Z-score)
plot_logo_zscore(
    rmats_df,
    col_prefix='5end',
    hue='Regulation',
    ylim=(-15,15),
    filename=f'{OUTPUT_PATH}/logo_5end.svg'
)


plot_logo_zscore(
    rmats_df,
    col_prefix='bp',
    hue='Regulation',
    ylim=(-15,15),
    filename=f'{OUTPUT_PATH}/logo_bp.svg'
)


plot_logo_zscore(
    rmats_df,
    col_prefix='3end',
    hue='Regulation',
    ylim=(-15,15),
    filename=f'{OUTPUT_PATH}/logo_3end.svg'
)

In [ ]:
# check distribution of numerical features
features = rmats_df.select_dtypes(include='number').columns.difference(['Regulation', 'bp_same'])
for col in features:
    plot_violin(
        df=rmats_df,
        column=col,
        hue='Regulation',
        filename=f'{OUTPUT_PATH}/violin_{col}.svg'
    )

#### 6. Save dataframe with sequence related features only

In [ ]:
# select features to be use in the machine learning model: only features retrived from primary sequence
ml_input_df = rmats_df.drop(columns=['siCTRL_expression', 'siFam32a_expression', 'Mean_IncLevel_siCTRL', 'Mean_IncLevel_siFam32a', 'delta_3ss_score'])
ml_input_df.to_csv(f'{OUTPUT_PATH}/ml_input.csv', index=False)